**Table of contents**<a id='toc0_'></a>    
- 1. [Setup](#toc1_)    
  - 1.1. [Load Dataset](#toc1_1_)    
  - 1.2. [Configuration](#toc1_2_)    
- 2. [Xây dựng Vocab](#toc2_)    
- 3. [Transformer](#toc3_)    
  - 3.1. [Chuẩn bị dataset](#toc3_1_)    
  - 3.2. [Chuẩn bị model](#toc3_2_)    
  - 3.3. [Training](#toc3_3_)    
- 4. [Thử nghiệm:](#toc4_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[Setup](#toc0_)

In [ ]:
import torch

print(torch.__version__)
print(torch.version.cuda)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

!nvidia-smi

In [ ]:
# Kiểm tra root_dir trên Kaggle
import os
print(os.listdir("/kaggle/input/datasets/nostagiguideus17"))


# Thiết lập để import source code
import sys
sys.path.append("/kaggle/input//nostagiguideus17/abstractive-summary-vers-transformer")

input_path = "/kaggle/input/datasets/nostagiguideus17/abstractive-summary-vers-transformer"

## 1.1. <a id='toc1_1_'></a>[Load Dataset](#toc0_)

In [ ]:
import pandas as pd
from datasets import Dataset

dataset = Dataset.from_parquet(input_path + '/data/train-00000-of-00001.parquet')
train_df = dataset.to_pandas()


dataset = Dataset.from_parquet(input_path + '/data/valid-00000-of-00001.parquet')
valid_df = dataset.to_pandas()


valid_df.head()

,article,summary
0,Giải thưởng công bố gần đây bởi World Travel A...,InterContinental Phu Quoc Long Beach Resort đã...
1,Theo bảng xếp hạng 20 quốc gia tốt nhất thế gi...,Việt Nam đã xếp hạng 15 trên bảng xếp hạng 20 ...
2,"Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ..."
3,Giải thưởng do Tạp chí du lịch Condé Nast Trav...,"Phú Quốc, đảo ngọc của Việt Nam, đã được vinh ..."
4,KKday Vietnam vừa công bố hợp tác chiến lược S...,KKday Vietnam vừa công bố hợp tác chiến lược v...


## 1.2. <a id='toc1_2_'></a>[Configuration](#toc0_)

In [17]:
from src.interfaces import ModelConfig
import torch

config = ModelConfig(
    lowercase = True, 
    dropout_prob = 0.1,

    vocab_size = 30000,
    embed_dim= 512,
    max_target_length = 1000,
)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

# 2. <a id='toc2_'></a>[Xây dựng Vocab](#toc0_)

In [18]:
# from src.preprocess import BPETokenizer

# tokenizer = BPETokenizer(config)

# tokenizer.train_from_iterator(df["article"].tolist() + df["summary"].tolist())

# tokenizer.save("/kaggle/working/tokenizer")

In [19]:
from src.preprocess import BPETokenizer

tokenizer = BPETokenizer.load(
    config = config,
    save_folder = input_path + "/model_save/ViT_tokenizer"
)

config.vocab_size = tokenizer.vocab_size()

[Info] Đã tải thành công Tokenizer từ: ./model_save/ViT_tokenizer


In [20]:
assert config.vocab_size == tokenizer.vocab_size(), "tokenizer sở hữu Vocab Size khác cấu hình hệ thống"

pad_id = tokenizer.token_to_id(tokenizer.special_tokens.pad_token)

print(f"Vocab Size: {config.vocab_size}")
print(f"ID của <UNK>: {tokenizer.token_to_id(tokenizer.special_tokens.unk_token)}")
print(f"ID của <PAD>: {tokenizer.token_to_id(tokenizer.special_tokens.pad_token)}")
print(f"ID của <EOS>: {tokenizer.token_to_id(tokenizer.special_tokens.eos_token)}")

Vocab Size: 36096
ID của <UNK>: 2
ID của <PAD>: 0
ID của <EOS>: 1


# 3. <a id='toc3_'></a>[Transformer](#toc0_)

## 3.1. <a id='toc3_1_'></a>[Chuẩn bị dataset](#toc0_)

In [21]:
from src.dataset import SummarizationDataset

train_dataset = SummarizationDataset(
    dataset=df,

    encode_fn=tokenizer.encode_batch, # Hàm mã hóa hàng loạt (batch encoding)
)

In [22]:
from torch.utils.data import DataLoader
from src.dataset import TransformerCollate


train_loader = DataLoader(
    train_dataset, 

    batch_size=16, 

    shuffle=True, 

    collate_fn= TransformerCollate(pad_id)
)

## 3.2. <a id='toc3_2_'></a>[Chuẩn bị model](#toc0_)

In [29]:
from src.optimus import Transformer

transformer:Transformer = Transformer(
    vocab_size = config.vocab_size,
    d_model    = config.embed_dim,
    pad_id     = pad_id,

    nhead           = 8,
    num_encoder_layers  = 2,
    num_decoder_layers  = 4,
    dim_feedforward     = config.embed_dim * 3,
    
    dropout             = config.dropout_prob,
    seq_max_len         = config.max_target_length
)

optimizer_state = None
scheduler_state = None
reload_state = False

In [30]:
from src.optimus import Transformer

transformer, optimizer_state, scheduler_state = Transformer.load_state(input_path + "/model_save/optimus_ViT_small")
reload_state = True


[Info] Đã tái tạo và tải trạng thái mô hình thành công từ: ./model_save/optimus_ViT_small
       - Tìm thấy trạng thái Optimizer.
       - Tìm thấy trạng thái Scheduler.


## 3.3. <a id='toc3_3_'></a>[Training](#toc0_)

In [ ]:
from torch import nn
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

epochs = 0
save_interval = 10
accumulation_steps = 4

transformer.to(device)
criterion = nn.CrossEntropyLoss(ignore_index=pad_id, label_smoothing=0.1)

loss_trajectory = []

# Load Optimizer
optimizer = AdamW(transformer.parameters(), lr=1e-4, weight_decay=0.01)


# Load scheduler
total_steps = epochs * len(train_loader)
warmup_steps = 0.1  # Dành 10% tổng số step đầu tiên để warmup (tăng dần LR từ 0)
    
scheduler = get_cosine_schedule_with_warmup(
    optimizer, 
    num_warmup_steps = int(warmup_steps*total_steps), 
    num_training_steps = total_steps
)


# Reload from previous saving state
if reload_state:
    optimizer.load_state_dict(optimizer_state)
    scheduler.load_state_dict(scheduler_state)
    



In [ ]:
from tqdm import tqdm

batches = 1

for epoch in range(epochs):
    transformer.train()
    total_loss = 0
        
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=True)
        
    for index, batch in enumerate(progress_bar):

        source_ids, target_ids = [x.to(device) for x in batch]
        
        # 1. Target_label -> Nhãn để tính loss
        target_label = target_ids.clone()    # Nhãn để tính loss

        # 2. Target_input (Right-Shift) -> feed cho decoder học 
        # Kẹp thẻ PAD vào đầu để làm Start Token
        start_tokens = torch.full((target_ids.size(0), 1), transformer.pad_id, dtype=torch.long, device=device)
        # Cắt bỏ thẻ EOS ở cuối ([:, :-1]), và 
        target_input = torch.cat([start_tokens, target_ids[:, :-1]], dim=1)

        # Truyền vào mô hình
        logits = transformer(source_ids=source_ids, target_ids=target_input)
            
        # Tính Loss
        loss = criterion(logits.view(-1, transformer.vocab_size), target_label.view(-1))
            
        loss.backward()
            
        # Cắt Gradient tránh bùng nổ
        if ((index + 1) % accumulation_steps == 0) or (index + 1 == len(train_loader)):
            torch.nn.utils.clip_grad_norm_(transformer.parameters(), max_norm=1.0)
            optimizer.step()  # Cập nhật Gradient 
            scheduler.step()  # Cập nhật learning rate
            optimizer.zero_grad()

        # Chỉnh record
        progress_bar.set_postfix(loss=loss.item())
            
        loss_trajectory.append(loss.item())
        total_loss += loss.item()

        batches -= 1
        if (batches == 0): break


            
    avg_loss = total_loss / len(train_loader)
    print(f"Hoàn thành Epoch {epoch+1} | Average Loss: {avg_loss:.4f}\n")

    if ((epoch+1) % save_interval == 0):
        loss_series = pd.Series(loss_trajectory[:])
        save_dir = f"./kaggle/working/model/checkpoint_{epoch+1}_epochs"
        
        transformer.save_state(save_dir, optimizer, scheduler)
        loss_series.to_csv(save_dir + "/loss", index=False)
        

Epoch 1/1:   0%|          | 0/674 [00:00<?, ?it/s, loss=416]

Hoàn thành Epoch 1 | Average Loss: 0.6179

[Info] Đã lưu model thành công tại: ./kaggle/working/model/checkpoint_1_epochs


In [ ]:
from src.visualize import visualizeLoss

# visualizeLoss(pd.Series(loss_trajectory[:]))

# 4. <a id='toc4_'></a>[Thử nghiệm:](#toc0_)

In [46]:
input_texts = valid_df['article'][:10].tolist()
expected_output = valid_df['summary'][:10].tolist()

# List chứa các câu tóm tắt sau khi model sinh ra
generated_summaries = []


transformer.eval() 
with torch.no_grad():
    for text in input_texts:
        # 1. Encode từng câu một và thêm chiều batch (batch_size = 1)
        # Giả sử tokenizer của bạn hỗ trợ truncate/pad, nếu không thì cứ encode bình thường
        source_ids = torch.tensor(tokenizer.encode(text), dtype=torch.long).unsqueeze(0).to(device)
        
        eos_id = tokenizer.token_to_id(tokenizer.special_tokens.eos_token)
        
        # 2. Sinh văn bản bằng Beam Search
        output_ids = transformer.beam_search(
            source_ids=source_ids,
            eos_id=eos_id,
            beam_width=4,
            device=device
        )
        
        # 3. Decode output
        # output_ids đang có dạng [1, seq_len], lấy phần tử đầu tiên -> [seq_len]
        decoded_text = tokenizer.decode(output_ids.cpu().numpy()[0])
        
        generated_summaries.append(decoded_text)

results = pd.DataFrame(
    {
        "article" : input_texts,
        "reference" : expected_output,
        "prediction" : generated_summaries
    }
)

results

,article,reference,prediction
0,Giải thưởng công bố gần đây bởi World Travel A...,InterContinental Phu Quoc Long Beach Resort đã...,<pad> Giải thưởng công bố gần đây bởi World Tr...
1,Theo bảng xếp hạng 20 quốc gia tốt nhất thế gi...,Việt Nam đã xếp hạng 15 trên bảng xếp hạng 20 ...,<pad> Condé Nast Trav đã công bố bảng xếp hạng...
2,"Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","<pad> Ngày hội Văn hóa, Thể thao và Du lịch cá..."
3,Giải thưởng do Tạp chí du lịch Condé Nast Trav...,"Phú Quốc, đảo ngọc của Việt Nam, đã được vinh ...","<pad> Giải thưởng ""Những hòn đảo tuyệt vời nhấ..."
4,KKday Vietnam vừa công bố hợp tác chiến lược S...,KKday Vietnam vừa công bố hợp tác chiến lược v...,<pad> KKday Vietnam vừa công bố hợp tác chiến ...
5,Công ty du lịch NhatbanAZ mở bán chùm tour Nhậ...,Công ty du lịch Nhật Bản AZ tung ra các tour d...,<pad> NhatbanAZ mở bán chùm tour Nhật Bản khởi...
6,Chùa Phổ Quang (chùa Xuân Lũng) nằm trên gò đấ...,"Chùa Phổ Quang, một công trình kiến trúc được ...",<pad> Chùa Phổ Quang (chùa Xuân Lũng) nằm trên...
7,Khám phá 'thiên đường' nghỉ dưỡng giữa mây ngà...,Bài viết giới thiệu hai điểm đến nghỉ dưỡng lý...,<pad> Du khách muốn đến miền Bắc 'săn mây' thì...
8,Du lịch mùa thu tại Hàn Quốc và Nhật Bản giúp ...,Tour liên tuyến Hàn Quốc - Nhật Bản (7-8 ngày)...,<pad> Du lịch mùa thu tại Hàn Quốc và Nhật Bản...
9,Bảo tàng thịt nướng (Museum of BQQ) mở cửa vào...,Bảo tàng thịt nướng (Museum of BQQ) sẽ mở cửa ...,<pad> Bảo tàng thịt nướng (Museum of BQQ) mở c...


In [35]:
device

device(type='cuda')